# Capítulo 10: Reamostragem

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 5 de James et al. (2023).

> **⚠️ Atenção — Em construção**
>
> A visão geral deste capítulo ainda será escrita.

## Seções

| Seção | Tópico |
|---|---|
| [10.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/01-o-conjunto-de-validacao.html) | O Conjunto de Validação |
| [10.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/02-leave-one-out.html) | Validação Cruzada Leave-One-Out |
| [10.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/03-validacao-cruzada-k-fold.html) | Validação Cruzada k-Fold |
| [10.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/04-vies-e-variancia-na-validacao-cruzada.html) | Viés e Variância na Validação Cruzada |
| [10.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/05-validacao-cruzada-em-classificacao.html) | Validação Cruzada em Classificação |
| [10.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/06-vazamento.html) | Vazamento: o Pré-processamento Dentro da Validação |

## O Conjunto de Validação

> **📌 Nota**
>
> Esta seção corresponde à seção 5.1.1 de James et al. (2023).

O número que decide entre dois modelos é o MSE de teste, definido na seção 7.6: o erro medido em observações que ficaram de fora do ajuste. Numa simulação, basta sortear mais pontos da mesma $f$. Com dado real quase nunca há um conjunto de teste separado de antemão; há uma tabela, e todas as linhas dela são candidatas a ajustar o modelo. A saída mais simples é fabricar o conjunto de teste: separar ao acaso uma parte das observações, ajustar o modelo no resto e medir o erro na parte separada.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

plt.style.use("estilo-figuras.mplstyle")

> **🔷 Conceito**
>
> A **abordagem do conjunto de validação** divide as observações disponíveis, ao acaso, em duas partes. O **conjunto de treino** ajusta o modelo; o **conjunto de validação** (em inglês, *hold-out set*) não participa do ajuste e serve só para medir o erro das previsões. O MSE sobre o conjunto de validação é uma estimativa do MSE de teste.

### Os carros e os polinômios

A pergunta de trabalho vem dos dados `Auto`: de que grau deve ser o polinômio em `potencia` que prevê `milhas_por_galao`? Na seção 8.5, o R² de treino subiu do grau 1 para o 2 e para o 5, como sobe sempre que o modelo ganha colunas, e por isso não servia para responder. O conjunto de validação serve.

In [ ]:
auto = pd.read_csv("dados/Auto.csv")
n_interrogacao = int((auto["potencia"] == "?").sum())
auto["potencia"] = pd.to_numeric(auto["potencia"], errors="coerce")
auto = auto.dropna(subset=["potencia"]).reset_index(drop=True)

X = auto[["potencia"]]
y = auto["milhas_por_galao"]

n_interrogacao, len(auto)

Cinco carros trazem `potencia` como o texto `"?"`; `pd.to_numeric(..., errors="coerce")` os transforma em `NaN` e `dropna` os descarta, e restam 392 carros.

In [ ]:
def modelo_polinomial(grau):
    return make_pipeline(
        StandardScaler(),
        PolynomialFeatures(grau, include_bias=False),
        LinearRegression(),
    )


graus = range(1, 11)

> **🔧 Função**
>
> **`make_pipeline(passo1, ..., estimador)`**: encadeia transformações e um estimador num objeto só, que se ajusta com `fit` e prevê com `predict` repetindo as mesmas transformações.
>
> **`StandardScaler()`**: deixa cada coluna com média 0 e desvio padrão 1.
>
> **`PolynomialFeatures(grau, include_bias=False)`**: troca a coluna $x$ por $x, x^2, \dots, x^{\text{grau}}$, sem a coluna de 1, que o `LinearRegression` já cobre com o intercepto.
>
> **`LinearRegression()`**: ajusta os coeficientes por mínimos quadrados.

`modelo_polinomial(grau)` devolve o modelo de um grau qualquer, e `graus` percorre de 1 a 10. O `StandardScaler` vem antes das potências pelo motivo da seção 8.5: elevar `potencia` à décima potência sem padronizar deixa as colunas em escalas tão distantes que a solução de mínimos quadrados perde precisão.

### Uma divisão

Metade dos 392 carros ajusta, a outra metade valida.

In [ ]:
X_treino, X_validacao, y_treino, y_validacao = train_test_split(
    X, y, test_size=196, random_state=10
)

mse_uma_divisao = np.array([
    mean_squared_error(
        y_validacao, modelo_polinomial(grau).fit(X_treino, y_treino).predict(X_validacao)
    )
    for grau in graus
])

len(X_treino), len(X_validacao)

> **🔧 Função**
>
> **`train_test_split(X, y, test_size, random_state)`**: sorteia quais linhas vão para cada lado e devolve, nesta ordem, `X` de treino, `X` de validação, `y` de treino e `y` de validação.
>
> - `test_size=196`: um número inteiro é a quantidade de linhas separadas, não uma fração.
> - `random_state=10`: a semente do sorteio, que deixa a divisão reproduzível.
>
> **`mean_squared_error(y_verdadeiro, y_previsto)`**: a média dos quadrados das diferenças entre os dois.

Cada grau é ajustado nos 196 carros de treino e julgado nos 196 de validação:

In [ ]:
pd.DataFrame(
    {"MSE de validação": [round(float(v), 2) for v in mse_uma_divisao]},
    index=pd.Index(graus, name="grau"),
)

In [ ]:
grau_minimo_uma_divisao = int(graus[np.argmin(mse_uma_divisao)])
ganho_grau_2 = round(float(mse_uma_divisao[0] - mse_uma_divisao[1]), 2)
amplitude_graus_2_a_4 = round(
    float(mse_uma_divisao[1:4].max() - mse_uma_divisao[1:4].min()), 2
)

grau_minimo_uma_divisao, ganho_grau_2, amplitude_graus_2_a_4

> **🔧 Função**
>
> **`np.argmin(arr, axis)`**: a posição do menor valor de `arr`. Sem `axis`, no array inteiro; com `axis=1`, a posição do menor valor em cada linha.

A reta erra 23,06; a parábola, 19,72. A curvatura derruba o erro em 3,34. Dali em diante a tabela é quase plana: entre o maior e o menor MSE dos graus 2, 3 e 4 a diferença é de 0,01, e o menor MSE de validação cai no grau 5, com 19,26, como aponta o `np.argmin`. Por esta divisão, o polinômio de grau 5 seria o escolhido. Mas a divisão foi um sorteio. O que acontece com essa escolha se os carros caírem de outro jeito?

### Dez divisões

A mesma conta, repetida com as sementes de 10 a 19: dez sorteios diferentes dos 196 carros de validação, dez curvas.

In [ ]:
sementes = range(10, 20)
mse_dez = []
for semente in sementes:
    X_tr, X_va, y_tr, y_va = train_test_split(
        X, y, test_size=196, random_state=semente
    )
    mse_dez.append([
        mean_squared_error(y_va, modelo_polinomial(grau).fit(X_tr, y_tr).predict(X_va))
        for grau in graus
    ])
mse_dez = np.array(mse_dez)

grau_minimo_por_divisao = [int(graus[i]) for i in np.argmin(mse_dez, axis=1)]
grau_2_abaixo_do_1_em_todas = bool((mse_dez[:, 1] < mse_dez[:, 0]).all())

grau_minimo_por_divisao, grau_2_abaixo_do_1_em_todas

O grau de menor MSE muda de divisão para divisão: 5, 2, 9, 7, 9, 5, 2, 9, 7 e 10, na ordem das sementes. Uma coisa não muda: `grau_2_abaixo_do_1_em_todas` confirma que, nas dez divisões, a parábola erra menos que a reta.

In [ ]:
amplitude_por_grau = mse_dez.max(axis=0) - mse_dez.min(axis=0)
pd.DataFrame(
    {"amplitude entre as dez divisões": [round(float(v), 2) for v in amplitude_por_grau]},
    index=pd.Index(graus, name="grau"),
)

A amplitude é o MSE de validação da pior divisão menos o da melhor, grau a grau. Para a reta, as dez estimativas se espalham por 7,53; para a parábola, por 7,49.

In [ ]:
# Figura: MSE de validação contra o grau do polinômio em potencia, nos dados Auto, com 196 carros de treino e 196 de validação. Esquerda: uma divisão (semente 10). Direita: dez divisões (sementes 10 a 19); a curva laranja é a mesma da esquerda.
fig, (ax_uma, ax_dez) = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)

ax_uma.plot(graus, mse_uma_divisao, color="C1", marker="o", linewidth=2)
ax_uma.set_title("uma divisão")

for linha in mse_dez[1:]:
    ax_dez.plot(graus, linha, color="C0", linewidth=1.2)
ax_dez.plot([], [], color="C0", linewidth=1.2, label="sementes 11 a 19")
ax_dez.plot(graus, mse_dez[0], color="C1", linewidth=2, label="semente 10")
ax_dez.set_title("dez divisões")
ax_dez.legend(loc="upper center")

for ax in (ax_uma, ax_dez):
    ax.set_xlabel("grau do polinômio")
    ax.set_xticks(list(graus))
ax_uma.set_ylabel("MSE de validação")
ax_uma.set_ylim(13, 29)
plt.tight_layout()
plt.show()

As dez curvas descem do grau 1 para o grau 2, cada uma numa altura própria. O painel direito não autoriza escolher entre o grau 2 e o grau 9, porque divisões diferentes discordam sobre isso. O que ele sustenta é mais modesto: a reta não basta.

### As duas desvantagens

A validação é fácil de entender e de programar, e cobra dois preços por isso.

**Variância alta.** A estimativa depende de quais carros caíram no treino e quais caíram na validação. A amplitude de 7,53 no grau 1 é esse preço medido: dez sorteios da mesma tabela, com o mesmo modelo, dão estimativas de erro que se espalham por mais de sete unidades de MSE.

**Viés para cima.** O modelo é ajustado com metade dos carros. Um modelo ajustado com menos dados tende a errar mais que o mesmo modelo ajustado com todos, e o erro medido na validação tende a superestimar o erro do modelo que se vai usar de fato, ajustado nos 392. O tamanho desse viés é medido na seção 10.4.

As duas desvantagens vêm da mesma escolha, a de separar uma metade fixa, uma vez só. A validação cruzada muda essa escolha: cada observação passa pelos dois lados, e os ajustes podem usar bem mais que a metade dos dados.

## Validação Cruzada Leave-One-Out

> **📌 Nota**
>
> Esta seção corresponde à seção 5.1.2 de James et al. (2023).

Em vez de separar metade das observações para validar, dá para separar **uma**. Ajusta-se o modelo nas $n-1$ restantes, prevê-se a que ficou de fora e anota-se o erro. Depois devolve-se essa observação, separa-se a seguinte e repete-se, até que cada uma das $n$ tenha ficado de fora exatamente uma vez.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

plt.style.use("estilo-figuras.mplstyle")

auto = pd.read_csv("dados/Auto.csv")
auto["potencia"] = pd.to_numeric(auto["potencia"], errors="coerce")
auto = auto.dropna(subset=["potencia"]).reset_index(drop=True)

X = auto[["potencia"]]
y = auto["milhas_por_galao"]


def modelo_polinomial(grau):
    return make_pipeline(
        StandardScaler(),
        PolynomialFeatures(grau, include_bias=False),
        LinearRegression(),
    )


graus = range(1, 11)

> **🔷 Conceito**
>
> A **validação cruzada *leave-one-out*** (LOOCV, "deixe um de fora") ajusta o modelo $n$ vezes. No ajuste $i$, a observação $(x_i, y_i)$ fica de fora, o modelo é ajustado nas outras $n-1$ e produz a previsão $\hat y_i$ para ela. O erro dessa previsão é $\mathrm{MSE}_i = (y_i - \hat y_i)^2$, e a estimativa do MSE de teste é a média dos $n$ erros:
>
> $$
> \mathrm{CV}_{(n)} = \frac{1}{n}\sum_{i=1}^n \mathrm{MSE}_i.
> $$

Um $\mathrm{MSE}_i$ sozinho é uma estimativa péssima do erro de teste: depende de uma observação só, e um carro atípico o joga lá para cima. A média de $n$ deles não tem esse problema.

A troca ataca as duas desvantagens do conjunto de validação. Cada ajuste usa $n-1$ observações, quase todas, e não metade; o modelo avaliado fica muito mais parecido com o que se vai usar de fato, ajustado em todas, e o viés para cima encolhe. E não há sorteio nenhum: cada observação sai uma vez, na ordem em que está, e rodar a LOOCV de novo dá exatamente o mesmo número.

### A LOOCV nos carros

A pergunta continua a mesma: que grau de polinômio em `potencia` prevê melhor `milhas_por_galao` nos dados `Auto`? Cada grau passa agora pela LOOCV.

In [ ]:
loocv = np.array([
    -cross_val_score(
        modelo_polinomial(grau), X, y,
        cv=LeaveOneOut(), scoring="neg_mean_squared_error",
    ).mean()
    for grau in graus
])

len(X), len(X) * len(graus)

> **🔧 Função**
>
> **`LeaveOneOut()`**: o esquema de divisão da LOOCV. Com $n$ linhas, gera $n$ divisões, cada uma deixando uma linha diferente de fora.
>
> **`cross_val_score(estimador, X, y, cv, scoring)`**: ajusta e avalia o estimador em cada divisão de `cv` e devolve um array com uma pontuação por divisão.
>
> - `cv=LeaveOneOut()`: as divisões a usar.
> - `scoring="neg_mean_squared_error"`: a pontuação é o MSE com o sinal trocado. O `scikit-learn` sempre maximiza a pontuação; por isso o erro entra negativo, e o `-` na frente de `cross_val_score` o desfaz.

São 392 carros, portanto 392 ajustes por grau, e 3.920 ajustes nos dez graus. A média de cada array de 392 erros é a $\mathrm{CV}_{(n)}$ daquele grau:

In [ ]:
pd.DataFrame(
    {"MSE da LOOCV": [round(float(v), 2) for v in loocv]},
    index=pd.Index(graus, name="grau"),
)

In [ ]:
grau_minimo_loocv = int(graus[np.argmin(loocv)])
salto_grau_1_para_2 = round(float(loocv[0] - loocv[1]), 2)
amplitude_graus_2_a_10 = round(float(loocv[1:].max() - loocv[1:].min()), 2)
ganho_minimo_sobre_grau_2 = round(float(loocv[1] - loocv.min()), 2)

grau_minimo_loocv, salto_grau_1_para_2, amplitude_graus_2_a_10, ganho_minimo_sobre_grau_2

A reta erra 24,23; a parábola, 19,25, e o erro cai 4,98 de um grau para o outro. Do grau 2 ao grau 10, as nove estimativas cabem numa faixa de 0,66. O menor MSE da LOOCV cai no grau 7, com 18,83, como aponta o `np.argmin`: 0,42 abaixo da parábola, contra os 4,98 que a curvatura já tinha tirado.

In [ ]:
# Figura: MSE da LOOCV contra o grau do polinômio em potencia, nos dados Auto. O eixo vertical é o mesmo da figura do conjunto de validação, de 13 a 29.
fig, ax = plt.subplots(figsize=(6, 4.2))
ax.plot(graus, loocv, color="C1", marker="o", linewidth=2)
ax.set_xlabel("grau do polinômio")
ax.set_ylabel("MSE da LOOCV")
ax.set_xticks(list(graus))
ax.set_ylim(13, 29)
plt.tight_layout()
plt.show()

É uma curva só, e ela não depende de semente. A figura da seção 10.1 mostrava dez curvas para as dez divisões sorteadas; aqui não há o que sortear, e a mesma tabela dá sempre a mesma curva. A leitura que ela sustenta é a mesma que as dez curvas sustentavam juntas: a reta não basta, e depois da parábola o ganho é pequeno.

### Um ajuste no lugar de 392

A LOOCV custa $n$ ajustes por modelo. Para regressão por mínimos quadrados, linear ou polinomial, existe um atalho que custa **um**:

$$
\mathrm{CV}_{(n)} = \frac{1}{n}\sum_{i=1}^n \left(\frac{y_i - \hat y_i}{1 - h_i}\right)^2,
$$

em que $\hat y_i$ agora vem do ajuste com todas as $n$ observações, e $h_i$ é a alavancagem da observação $i$, a diagonal da matriz chapéu definida na seção 8.6. Sem o denominador, a soma seria a média dos quadrados dos resíduos de treino. O $1 - h_i$ corrige isso ponto a ponto: o resíduo de um ponto de alavancagem alta é inflado na medida exata do quanto aquele ponto puxou o próprio ajuste para perto de si. O que acontece com o termo de uma observação cujo $h_i$ se aproxima de 1?

A conferência é fazer as duas contas e comparar. Para a fórmula, o chunk ajusta o modelo nos 392 carros, reconstrói as colunas que o `LinearRegression` de fato recebe (padronização seguida das potências), acrescenta a coluna de 1 e calcula a matriz chapéu como na seção 8.6.

In [ ]:
formula = []
for grau in graus:
    previsto = modelo_polinomial(grau).fit(X, y).predict(X)
    colunas = make_pipeline(
        StandardScaler(), PolynomialFeatures(grau, include_bias=False)
    ).fit_transform(X)
    X_design = np.column_stack([np.ones(len(X)), colunas])
    H = X_design @ np.linalg.inv(X_design.T @ X_design) @ X_design.T
    h = np.diag(H)
    formula.append(np.mean(((y - previsto) / (1 - h)) ** 2))
formula = np.array(formula)

pd.DataFrame(
    {"LOOCV (392 ajustes)": loocv.round(6), "fórmula (1 ajuste)": formula.round(6)},
    index=pd.Index(graus, name="grau"),
)

> **🔧 Função**
>
> **`fit_transform(X)`**: ajusta as transformações de um `Pipeline` sem estimador final e devolve o resultado delas sobre `X`. Aqui, as colunas padronizadas $x, x^2, \dots, x^{\text{grau}}$.
>
> **`np.ones(n)`**: um array de `n` uns, a coluna do intercepto. **`np.column_stack(lista)`**: junta os arrays da lista lado a lado, como colunas de uma matriz.
>
> **`np.linalg.inv(M)`**: a inversa da matriz `M`. **`np.diag(M)`**: a diagonal de uma matriz quadrada, como array; aqui, os $h_i$.

In [ ]:
formula_bate_com_loocv = bool(np.allclose(loocv, formula))
diferenca = np.abs(loocv - formula)
maior_diferenca = f"{diferenca.max():.1e}"
grau_da_maior_diferenca = int(graus[np.argmax(diferenca)])

formula_bate_com_loocv, maior_diferenca, grau_da_maior_diferenca

> **🔧 Função**
>
> **`np.allclose(a, b)`**: `True` se os dois arrays são iguais elemento a elemento, a menos de uma tolerância pequena para o arredondamento de ponto flutuante.

`formula_bate_com_loocv` confirma `True`: nos dez graus, um ajuste só dá o mesmo número que 392. A maior diferença entre as duas colunas é de $8{,}8 \times 10^{-8}$, no grau 10. Ela aparece no grau mais alto porque a matriz $X^\top X$ de um polinômio de grau alto é difícil de inverter com precisão, não porque a fórmula deixe de valer.

Há uma pergunta que o leitor atento faz aqui. Dentro da LOOCV, o `StandardScaler` é reajustado a cada um dos 392 ajustes, com a média e o desvio padrão de 391 carros; na fórmula, a padronização usa os 392. Por que a igualdade sobrevive? Porque padronizar é trocar $x$ por $(x - m)/s$, e um polinômio de grau $d$ em $(x - m)/s$ é, depois de expandido, um polinômio de grau $d$ em $x$. Os mínimos quadrados escolhem a melhor curva entre os mesmos polinômios, com ou sem a troca, e chegam à mesma curva; só os coeficientes que a descrevem mudam. A frase explica, e o `True` impresso acima prova.

O atalho é uma propriedade dos mínimos quadrados. Para a regressão logística, o *k*-NN e os outros métodos, a LOOCV não tem fórmula equivalente e exige mesmo os $n$ ajustes.

## Validação Cruzada k-Fold

> **📌 Nota**
>
> Esta seção corresponde à seção 5.1.3 de James et al. (2023).

Entre separar metade das observações uma vez e separar uma de cada vez $n$ vezes há um meio-termo. Divide-se a tabela em alguns grupos, e cada grupo, na sua vez, fica de fora enquanto os outros ajustam o modelo.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, LeaveOneOut, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

plt.style.use("estilo-figuras.mplstyle")

auto = pd.read_csv("dados/Auto.csv")
auto["potencia"] = pd.to_numeric(auto["potencia"], errors="coerce")
auto = auto.dropna(subset=["potencia"]).reset_index(drop=True)

X = auto[["potencia"]]
y = auto["milhas_por_galao"]


def modelo_polinomial(grau):
    return make_pipeline(
        StandardScaler(),
        PolynomialFeatures(grau, include_bias=False),
        LinearRegression(),
    )


graus = range(1, 11)

> **🔷 Conceito**
>
> A **validação cruzada *k-fold*** divide as $n$ observações, ao acaso, em $k$ grupos (em inglês, *folds*) de tamanho parecido. No passo $i$, o grupo $i$ é o conjunto de validação: o modelo é ajustado nos outros $k-1$ grupos e o erro das previsões no grupo $i$ é $\mathrm{MSE}_i$. A estimativa do MSE de teste é a média dos $k$ erros:
>
> $$
> \mathrm{CV}_{(k)} = \frac{1}{k}\sum_{i=1}^k \mathrm{MSE}_i.
> $$

A LOOCV é o caso $k = n$: $n$ grupos de uma observação cada. Na prática, usa-se $k = 5$ ou $k = 10$.

### Três maneiras de separar

A figura reúne os três esquemas, com dez observações. Cada linha é um ajuste; em cada linha, as células azuis ajustam o modelo e as laranja o julgam.

In [ ]:
# Figura: Os três esquemas de divisão com dez observações. Cada linha é um ajuste; azul é treino, laranja é validação. Conjunto de validação: um ajuste, metade das observações de fora. LOOCV: dez ajustes, uma observação de fora em cada. 5-fold: cinco ajustes, um quinto das observações de fora em cada.
n_ilustra = 10


def desenha_esquema(ax, validacao_por_ajuste, titulo):
    n_ajustes = len(validacao_por_ajuste)
    for linha, validacao in enumerate(validacao_por_ajuste):
        cores = ["C1" if j in validacao else "C0" for j in range(n_ilustra)]
        ax.barh(
            [n_ajustes - 1 - linha] * n_ilustra, [0.9] * n_ilustra,
            left=np.arange(n_ilustra), height=0.8, color=cores, align="edge",
        )
    ax.set_xlim(0, n_ilustra)
    ax.set_ylim(0, n_ajustes)
    ax.set_title(titulo, loc="left")
    ax.set_axis_off()


esquemas = [
    ([range(5, 10)], "conjunto de validação"),
    ([[j] for j in range(n_ilustra)], "LOOCV"),
    ([range(2 * i, 2 * i + 2) for i in range(5)], "5-fold"),
]

fig, eixos = plt.subplots(
    3, 1, figsize=(6.4, 5.6), gridspec_kw={"height_ratios": [1, 10, 5]}
)
for ax, (validacao_por_ajuste, titulo) in zip(eixos, esquemas):
    desenha_esquema(ax, validacao_por_ajuste, titulo)
fig.legend(
    handles=[Patch(color="C0", label="treino"), Patch(color="C1", label="validação")],
    loc="lower center", ncol=2,
)
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()

> **🔧 Função**
>
> **`ax.barh(y, width, left, height, color, align)`**: desenha barras horizontais, uma para cada altura em `y`. Aqui, cada barra é uma célula de uma faixa.
>
> - `width`: o comprimento de cada barra; 0,9 deixa um espaço entre células vizinhas.
> - `left`: onde cada barra começa no eixo horizontal.
> - `height`: a espessura da barra; menor que 1, deixa um espaço entre as faixas.
> - `color`: uma cor por barra.
> - `align="edge"`: a barra começa na altura `y`, em vez de se centrar nela.
>
> **`ax.set_axis_off()`**: esconde os eixos, as marcas e a malha, e deixa só o que foi desenhado.
>
> **`Patch(color, label)`**: um retângulo de cor, usado aqui só para dar à legenda uma entrada por cor.

No conjunto de validação, cada observação está de um lado só. Na LOOCV e no 5-fold, toda observação passa uma vez pela validação e, em todos os outros ajustes, pelo treino. A diferença entre os dois está no número de linhas: $n$ ajustes contra $k$. Num 10-fold com $n$ observações, quantas cada ajuste vê?

Esse número é o preço. Para regressão por mínimos quadrados, a fórmula da seção 10.2 faz a LOOCV com um ajuste só; para quase todos os outros métodos ela não existe, e a LOOCV exige mesmo os $n$ ajustes. Com um método que demore para ajustar, ou com $n$ grande, a diferença decide o que é viável.

### O k-fold nos carros

A pergunta de trabalho continua a mesma: que grau de polinômio em `potencia` prevê melhor `milhas_por_galao` nos dados `Auto`? No `scikit-learn`, os grupos do *k*-fold saem de um `KFold`, que vai no argumento `cv` do `cross_val_score`.

In [ ]:
dez_grupos = KFold(10, shuffle=True, random_state=10)
cv_grau_1 = -cross_val_score(
    modelo_polinomial(1), X, y, cv=dez_grupos, scoring="neg_mean_squared_error"
).mean()

len(X), round(float(cv_grau_1), 2)

> **🔧 Função**
>
> **`KFold(n_splits, shuffle, random_state)`**: o esquema de divisão do *k*-fold.
>
> - `n_splits=10`: o número de grupos, o $k$.
> - `shuffle=True`: embaralha as linhas antes de cortar os grupos. Sem ele, o primeiro grupo são as primeiras linhas da tabela, o segundo as seguintes, e assim por diante.
> - `random_state=10`: a semente do embaralhamento.

Com 392 carros, a LOOCV do grau 1 custa 392 ajustes; o 10-fold custa 10. O erro estimado para a reta é 24,13.

O `shuffle=True` não é detalhe. A tabela `Auto` não está em ordem aleatória:

In [ ]:
cv_grau_1_sem_embaralhar = -cross_val_score(
    modelo_polinomial(1), X, y, cv=KFold(10), scoring="neg_mean_squared_error"
).mean()

bool(auto["ano"].is_monotonic_increasing), round(float(cv_grau_1_sem_embaralhar), 2)

Os carros estão em ordem de `ano`, do mais antigo ao mais novo. Sem embaralhar, cada grupo é uma época, e o modelo que o julga foi ajustado quase só com carros de outros anos. O erro estimado da reta sobe de 24,13 para 27,44. A mesma coisa acontece com qualquer tabela ordenada por data, por cliente ou por região, e nenhum aviso aparece na tela. O que deveria mudar na conta se o objetivo fosse, de propósito, prever carros de anos que o modelo não viu?

### Nove partições

O embaralhamento é um sorteio, e outra semente dá outros grupos. O 10-fold, repetido com as sementes de 10 a 18, dá nove curvas.

In [ ]:
cv_nove = np.array([
    [
        -cross_val_score(
            modelo_polinomial(grau), X, y,
            cv=KFold(10, shuffle=True, random_state=semente),
            scoring="neg_mean_squared_error",
        ).mean()
        for grau in graus
    ]
    for semente in range(10, 19)
])

grau_minimo_por_particao = [int(graus[i]) for i in np.argmin(cv_nove, axis=1)]
todas_escolhem_o_grau_7 = set(grau_minimo_por_particao) == {7}

grau_minimo_por_particao, todas_escolhem_o_grau_7

`todas_escolhem_o_grau_7` confirma que as nove partições apontam o mesmo grau de menor MSE estimado, o 7. Para comparar a dispersão com a do conjunto de validação, o chunk abaixo refaz as dez divisões ao meio da seção 10.1, com as mesmas sementes de 10 a 19, e mede a amplitude grau a grau nos dois esquemas: o maior MSE estimado menos o menor.

In [ ]:
mse_dez_divisoes = []
for semente in range(10, 20):
    X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=196, random_state=semente)
    mse_dez_divisoes.append([
        mean_squared_error(y_va, modelo_polinomial(grau).fit(X_tr, y_tr).predict(X_va))
        for grau in graus
    ])
mse_dez_divisoes = np.array(mse_dez_divisoes)

amplitude_validacao = mse_dez_divisoes.max(axis=0) - mse_dez_divisoes.min(axis=0)
amplitude_kfold = cv_nove.max(axis=0) - cv_nove.min(axis=0)

pd.DataFrame(
    {
        "validação (10 divisões)": [round(float(v), 2) for v in amplitude_validacao],
        "10-fold (9 partições)": [round(float(v), 2) for v in amplitude_kfold],
    },
    index=pd.Index(graus, name="grau"),
)

In [ ]:
kfold_menos_disperso_em_todos_os_graus = bool((amplitude_kfold < amplitude_validacao).all())
kfold_menos_disperso_em_todos_os_graus

`kfold_menos_disperso_em_todos_os_graus` confirma que, nos dez graus, as nove estimativas do 10-fold se espalham menos que as dez do conjunto de validação. No grau 2, a validação espalha 7,49 e o 10-fold, 0,15; no grau 10, 8,88 contra 1,15.

A curva da LOOCV, que não depende de sorteio, entra na figura para comparação:

In [ ]:
loocv = np.array([
    -cross_val_score(
        modelo_polinomial(grau), X, y, cv=LeaveOneOut(), scoring="neg_mean_squared_error"
    ).mean()
    for grau in graus
])

loocv_dentro_das_nove = bool(
    ((loocv >= cv_nove.min(axis=0)) & (loocv <= cv_nove.max(axis=0))).all()
)

int(graus[np.argmin(loocv)]), loocv_dentro_das_nove

In [ ]:
# Figura: MSE estimado contra o grau do polinômio em potencia, nos dados Auto. As curvas azuis são o 10-fold com nove partições diferentes (sementes 10 a 18); a curva laranja tracejada é a LOOCV. O eixo vertical é o mesmo da figura do conjunto de validação, de 13 a 29.
fig, ax = plt.subplots(figsize=(6, 4.2))
for linha in cv_nove:
    ax.plot(graus, linha, color="C0", linewidth=1.2)
ax.plot([], [], color="C0", linewidth=1.2, label="10-fold (9 partições)")
ax.plot(graus, loocv, color="C1", linewidth=2, linestyle="--", label="LOOCV")
ax.set_xlabel("grau do polinômio")
ax.set_ylabel("MSE estimado")
ax.set_xticks(list(graus))
ax.set_xlim(0.5, 10.5)
ax.set_ylim(13, 29)
ax.legend(loc="upper center")
plt.tight_layout()
plt.show()

O eixo vertical é o da figura da seção 10.1, e a diferença de dispersão entre as duas se vê sem ler números: lá, dez curvas em alturas bem diferentes; aqui, nove curvas quase sobrepostas. A LOOCV também tem o mínimo no grau 7, e `loocv_dentro_das_nove` confirma que, em todos os graus, ela fica entre a menor e a maior das nove estimativas do 10-fold. No 10-fold, todo carro é julgado uma vez, por um modelo ajustado com os outros nove décimos da tabela; trocar a partição muda só quais carros se juntam em cada grupo, e isso mexe pouco na média.

### A validação cruzada contra o teste verdadeiro

Nos carros, não há como saber se a validação cruzada acertou: o MSE de teste verdadeiro é desconhecido. Com dado simulado, ele é conhecido, porque dá para sortear quantos pontos novos se quiser da mesma $f$.

São três cenários. O primeiro é o da seção 7.6: $f(x) = 4 + 0{,}3x + 3\sin x$, com ruído de desvio padrão 1,5. O segundo quase não tem curva; o terceiro tem uma curva mais apertada e pouco ruído. Em cada um, 100 pontos de treino, 20.000 de teste, polinômios de grau 1 a 10, e três curvas: o MSE de teste, a LOOCV e o 10-fold, as duas últimas calculadas só com os 100 pontos de treino.

In [ ]:
cenarios = {
    "seno": (lambda x: 4 + 0.3 * x + 3 * np.sin(x), 1.5),
    "quase linear": (lambda x: 4 + 0.3 * x + 0.3 * np.sin(x), 1.5),
    "curva forte, pouco ruído": (lambda x: 4 + 0.3 * x + 3 * np.sin(1.3 * x), 0.5),
}

rng = np.random.default_rng(10)
curvas = {}
for nome, (f, desvio) in cenarios.items():
    x_treino = rng.uniform(0, 10, size=100).reshape(-1, 1)
    y_treino = f(x_treino.ravel()) + rng.normal(0, desvio, size=100)
    x_teste = rng.uniform(0, 10, size=20_000).reshape(-1, 1)
    y_teste = f(x_teste.ravel()) + rng.normal(0, desvio, size=20_000)

    teste, cv_loo, cv_10 = [], [], []
    for grau in graus:
        modelo = modelo_polinomial(grau)
        teste.append(mean_squared_error(y_teste, modelo.fit(x_treino, y_treino).predict(x_teste)))
        cv_loo.append(-cross_val_score(
            modelo, x_treino, y_treino, cv=LeaveOneOut(), scoring="neg_mean_squared_error"
        ).mean())
        cv_10.append(-cross_val_score(
            modelo, x_treino, y_treino, cv=KFold(10, shuffle=True, random_state=10),
            scoring="neg_mean_squared_error",
        ).mean())
    curvas[nome] = {
        "teste": np.array(teste), "LOOCV": np.array(cv_loo), "10-fold": np.array(cv_10)
    }

> **🔧 Função**
>
> **`arr.reshape(-1, 1)`**: transforma um array de uma dimensão numa matriz de uma coluna, a forma que o `fit` espera para `X`. O `-1` deixa o `numpy` calcular o número de linhas.
>
> **`arr.ravel()`**: o caminho de volta, a matriz de uma coluna achatada num array de uma dimensão.

In [ ]:
# Figura: MSE de teste verdadeiro (azul), LOOCV (cinza tracejado) e 10-fold (laranja) contra o grau do polinômio, em três cenários simulados com 100 pontos de treino. O x marca o mínimo de cada curva. Cada painel tem a sua escala vertical.
fig, eixos = plt.subplots(1, 3, figsize=(11, 3.8))
estilos = {
    "teste": {"color": "C0", "linestyle": "-", "label": "teste"},
    "LOOCV": {"color": "0.45", "linestyle": "--", "label": "LOOCV"},
    "10-fold": {"color": "C1", "linestyle": "-", "label": "10-fold"},
}
for ax, (nome, curvas_do_cenario) in zip(eixos, curvas.items()):
    for chave, valores in curvas_do_cenario.items():
        ax.plot(graus, valores, linewidth=2, **estilos[chave])
        i = int(np.argmin(valores))
        ax.plot(
            graus[i], valores[i], marker="x", markersize=10, markeredgewidth=2.5,
            color=estilos[chave]["color"], linestyle="none", label="_nolegend_",
        )
    ax.set_title(nome)
    ax.set_xlabel("grau do polinômio")
    ax.set_xticks(list(graus))
    ax.set_xlim(0.5, 10.5)
eixos[0].set_ylabel("MSE")
eixos[0].legend()
plt.tight_layout()
plt.show()

A tabela resume o que a figura mostra: onde cai o mínimo de cada curva, e quanto se perde, em MSE de teste, por usar o grau que a validação cruzada escolheu em vez do grau que o teste escolheria.

In [ ]:
linhas = {}
for nome, c in curvas.items():
    grau_teste = int(np.argmin(c["teste"]))
    grau_loo = int(np.argmin(c["LOOCV"]))
    grau_10 = int(np.argmin(c["10-fold"]))
    linhas[nome] = {
        "grau: teste": grau_teste + 1,
        "grau: LOOCV": grau_loo + 1,
        "grau: 10-fold": grau_10 + 1,
        "MSE de teste mínimo": round(float(c["teste"][grau_teste]), 2),
        "perda: LOOCV": round(float(c["teste"][grau_loo] - c["teste"][grau_teste]), 2),
        "perda: 10-fold": round(float(c["teste"][grau_10] - c["teste"][grau_teste]), 2),
    }
pd.DataFrame.from_dict(linhas, orient="index")

In [ ]:
nivel = {}
for nome, c in curvas.items():
    as_duas_abaixo = (c["LOOCV"] < c["teste"]) & (c["10-fold"] < c["teste"])
    nivel[nome] = {
        "LOOCV acima": int(np.sum(c["LOOCV"] > c["teste"])),
        "10-fold acima": int(np.sum(c["10-fold"] > c["teste"])),
        "as duas abaixo": ", ".join(str(g) for g, b in zip(graus, as_duas_abaixo) if b),
        "maior distância": round(float(np.max(np.abs(c["10-fold"] - c["teste"]))), 2),
        "no grau": int(graus[np.argmax(np.abs(c["10-fold"] - c["teste"]))]),
    }
pd.DataFrame.from_dict(nivel, orient="index")

As três primeiras colunas contam graus, de 1 a 10: em quantos cada curva de validação cruzada fica acima do teste, e em quais as duas ficam abaixo dele. As duas últimas dão a maior distância entre o 10-fold e o teste, e o grau em que ela acontece.

O **nível** das curvas de validação cruzada erra, e erra de jeitos diferentes. No cenário do seno, as duas ficam acima do teste nos dez graus, e o 10-fold chega a se afastar 1,57 dele, no grau 1. No cenário quase linear, também ficam acima nos dez graus, mas mais perto: a maior distância do 10-fold ao teste é 0,18. Na curva forte com pouco ruído, as duas ficam abaixo do teste nos graus 1 a 4, 6, 8 e 9, e a maior distância, 0,45, está no grau 1.

O **ponto de mínimo** erra pouco. No cenário quase linear e no da curva forte, as três curvas apontam o mesmo grau, e a perda é zero. No seno, o teste aponta o grau 8, a LOOCV o 9 e o 10-fold o 6; usar o grau da LOOCV custa 0,17 de MSE de teste, e o do 10-fold, 0,04, sobre um mínimo de 2,56.

São três amostras, e é isso o que três amostras mostram. Às vezes o que interessa é o valor do erro de teste, porque se quer saber quão bem um modelo vai prever; aí o nível importa, e ele pode errar para qualquer lado. Mas, para escolher entre modelos, o que interessa é **onde está o mínimo**, e para isso as curvas de validação cruzada serviram nos três cenários.

## Viés e Variância na Validação Cruzada

> **📌 Nota**
>
> Esta seção corresponde à seção 5.1.4 de James et al. (2023).

A validação, o 5-fold, o 10-fold e a LOOCV dão quatro estimativas do mesmo número, o MSE de teste. Qual delas é melhor? Uma estimativa pode errar de dois jeitos, os mesmos dois da decomposição da seção 7.6, agora aplicados à estimativa em vez da previsão. Ela pode ter **viés**: sobre muitas amostras, a média dela passa do alvo. E pode ter **variância**: de uma amostra para outra, ela oscila em torno da própria média.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, LeaveOneOut, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler


def modelo_polinomial(grau):
    return make_pipeline(
        StandardScaler(),
        PolynomialFeatures(grau, include_bias=False),
        LinearRegression(),
    )

### O que o argumento prevê

**O viés.** Numa amostra de $n$ observações, cada ajuste da validação vê metade delas; cada ajuste do 5-fold, quatro quintos; do 10-fold, nove décimos; da LOOCV, $n - 1$. O modelo que se vai usar de fato é ajustado com as $n$. Um modelo ajustado com menos dados tende a errar mais, e então a estimativa mede o erro de um modelo pior que o que será usado: passa do alvo. Quanto menos dado cada ajuste vê, mais ela deve passar. O argumento prevê uma ordem: a LOOCV com o menor viés, depois o 10-fold, o 5-fold, e a validação com o maior.

**A variância.** Aqui há uma razão para esperar que a LOOCV perca. Dois ajustes quaisquer da LOOCV têm em comum todas as observações de treino menos uma, e por isso são quase o mesmo modelo; os $n$ erros dos quais ela tira a média são muito parecidos entre si, e sobem ou descem juntos quando a amostra muda. A média de quantidades muito correlacionadas oscila mais que a de quantidades pouco correlacionadas: no extremo, se as $n$ fossem iguais, a média não amorteceria nada. Os ajustes do 10-fold dividem uma parte menor do treino, e os erros deles, menos correlacionados, deveriam se compensar melhor.

Há uma razão na direção contrária. Cada ajuste da LOOCV vê mais dados que um do 10-fold, e um modelo ajustado com mais dados muda menos de amostra para amostra. Os dois efeitos competem, e o argumento sozinho não diz qual pesa mais. O que se pode fazer é medir.

### A medição

A medição usa o gerador da seção 7.6, $f(x) = 4 + 0{,}3x + 3\sin x$ com ruído de desvio padrão 1,5, e um modelo fixo, o polinômio de grau 4. Primeiro se sorteiam 20.000 pontos de teste. Depois, 200 amostras de treino de 100 pontos cada. Em cada amostra, o modelo ajustado nos 100 pontos é julgado nos 20.000, o que dá o MSE de teste verdadeiro dele, e as quatro estimativas são calculadas só com os 100 pontos.

In [ ]:
def f(x):
    return 4 + 0.3 * x + 3 * np.sin(x)


desvio = 1.5
n = 100
n_amostras = 200
grau = 4


def cv(x, y, esquema):
    return -cross_val_score(
        modelo_polinomial(grau), x, y, cv=esquema, scoring="neg_mean_squared_error"
    ).mean()


rng = np.random.default_rng(10)
x_teste = rng.uniform(0, 10, size=20_000).reshape(-1, 1)
y_teste = f(x_teste.ravel()) + rng.normal(0, desvio, size=20_000)

linhas = []
for r in range(n_amostras):
    x = rng.uniform(0, 10, size=n).reshape(-1, 1)
    y = f(x.ravel()) + rng.normal(0, desvio, size=n)
    x_tr, x_va, y_tr, y_va = train_test_split(x, y, test_size=n // 2, random_state=r)
    linhas.append({
        "teste": mean_squared_error(y_teste, modelo_polinomial(grau).fit(x, y).predict(x_teste)),
        "LOOCV": cv(x, y, LeaveOneOut()),
        "10-fold": cv(x, y, KFold(10, shuffle=True, random_state=r)),
        "5-fold": cv(x, y, KFold(5, shuffle=True, random_state=r)),
        "validação": mean_squared_error(
            y_va, modelo_polinomial(grau).fit(x_tr, y_tr).predict(x_va)
        ),
    })
medidas = pd.DataFrame(linhas)

medidas.shape

Cada linha de `medidas` é uma amostra; cada coluna, o MSE de teste verdadeiro ou uma das quatro estimativas dele. A partição do *k*-fold e da validação muda de amostra para amostra, com a semente `r` da réplica. Para cada coluna, a média sobre as 200 amostras e o desvio padrão:

In [ ]:
resumo = medidas.agg(["mean", "std"]).round(3)
resumo.index = ["média", "desvio padrão"]
resumo

> **🔧 Função**
>
> **`df.agg(lista)`**: aplica a cada coluna de `df` cada função da lista e devolve uma tabela com uma linha por função. Aqui, `"mean"` é a média e `"std"` o desvio padrão.

As duas comparações que interessam, conferidas sobre a tabela:

In [ ]:
estimativas = ["LOOCV", "10-fold", "5-fold", "validação"]
medias = medidas[estimativas].mean()
desvios = medidas[estimativas].std()

vies_cresce_quando_o_treino_encolhe = bool((np.diff(medias.to_numpy()) > 0).all())
loocv_oscila_menos_que_10_fold = bool(desvios["LOOCV"] < desvios["10-fold"])
validacao_oscila_mais_que_as_outras = bool(
    (desvios["validação"] > desvios.drop("validação")).all()
)

(
    vies_cresce_quando_o_treino_encolhe,
    loocv_oscila_menos_que_10_fold,
    validacao_oscila_mais_que_as_outras,
)

> **🔧 Função**
>
> **`np.diff(arr)`**: as diferenças entre elementos vizinhos, `arr[1] - arr[0]`, `arr[2] - arr[1]` e assim por diante. Todas positivas quer dizer que `arr` cresce do começo ao fim.

### O que a medição mostra

**O viés sai na ordem que o argumento previu.** `vies_cresce_quando_o_treino_encolhe` confirma que as médias crescem da LOOCV para a validação, na ordem em que os ajustes veem menos dados. Contra a média de 2,771 do MSE de teste verdadeiro, a LOOCV passa do alvo em 0,021 (2,792 menos 2,771), o 10-fold em 0,035 (2,806 menos 2,771), o 5-fold em 0,072 (2,843 menos 2,771) e a validação em 0,225 (2,996 menos 2,771). É o viés para cima da validação que a seção 10.1 enunciou sem medir; aqui ele tem tamanho, e é o maior dos quatro.

**A validação simples oscila muito mais que a validação cruzada.** `validacao_oscila_mais_que_as_outras` confirma que o desvio padrão dela, 0,714, passa dos três outros. Contra o do 5-fold, 0,437, é cerca de 1,6 vez maior (0,714 dividido por 0,437). É a variância alta da seção 10.1, agora medida contra as alternativas.

Vale olhar também a primeira coluna. O MSE de teste verdadeiro muda pouco de amostra para amostra, com desvio padrão de 0,087; a LOOCV oscila com 0,404, cerca de 4,6 vezes mais (0,404 dividido por 0,087). Com 100 pontos, mesmo a LOOCV erra, numa amostra só, bem mais do que o alvo varia.

**A LOOCV não oscilou mais que o 10-fold.** `loocv_oscila_menos_que_10_fold` confirma o contrário: 0,404 contra 0,418. O argumento da correlação é real, mas compete com o outro efeito, o de cada ajuste da LOOCV ver mais dados e ser mais estável, e nesta simulação o segundo pesou mais. A diferença é pequena, de 0,014 (0,418 menos 0,404). O que esta simulação mostra é que o argumento da correlação, sozinho, não basta para prever qual das duas oscila mais. Se a LOOCV tem o menor viés e, aqui, não oscilou mais, por que alguém escolheria o 10-fold?

### O que sustenta k = 5 ou k = 10

O custo. Nesta simulação, a LOOCV fez 100 ajustes por amostra, um por observação; o 10-fold fez 10, e o 5-fold, 5. Em troca de fazer dez vezes menos ajustes que a LOOCV, o 10-fold paga um viés a mais de 0,014 (2,806 menos 2,792), contra os 0,225 de viés da validação simples. Com um modelo que leva minutos para ajustar, ou com $n$ na casa das centenas de milhares, essa troca decide o que é viável. Com $n = 100.000$ e um modelo que leva um minuto para ajustar, quanto tempo levaria a LOOCV, e quanto o 10-fold?

## Validação Cruzada em Classificação

> **📌 Nota**
>
> Esta seção corresponde à seção 5.1.5 de James et al. (2023).

Quando a resposta é uma classe, a validação cruzada funciona do mesmo jeito que na regressão: separa-se uma parte das observações, ajusta-se o modelo no resto e mede-se o erro na parte separada. Só a régua muda. No lugar do MSE entra a taxa de erro, a fração de observações classificadas na classe errada.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

plt.style.use("estilo-figuras.mplstyle")

> **🔷 Conceito**
>
> Na LOOCV de um classificador, cada observação $i$ fica de fora uma vez, e o erro dela é $\mathrm{Err}_i = I(y_i \neq \hat y_i)$, a variável indicadora da seção 7.7: 1 se o modelo ajustado sem ela erra a sua classe, 0 se acerta. A estimativa da taxa de erro de teste é a média:
>
> $$
> \mathrm{CV}_{(n)} = \frac{1}{n}\sum_{i=1}^n \mathrm{Err}_i.
> $$
>
> No *k*-fold, $\mathrm{Err}_i$ passa a ser a taxa de erro no grupo $i$, e $\mathrm{CV}_{(k)}$ é a média das $k$ taxas. No conjunto de validação, é a taxa de erro na metade separada.

Um aviso sobre o nome. Nesta seção, como na 7.7, *k* é o número de vizinhos do *k*-NN. A validação cruzada usada aqui é o 10-fold, e os grupos dela são chamados sempre de "10 grupos", para as duas letras não se confundirem.

### Os pontos da seção 7.7

O dado é o simulado da seção 7.7: dois preditores uniformes em $[0, 10]$, a classe sorteada com probabilidade $\Pr(Y = 1 \mid X = x) = 1/(1 + e^{-(3\sin(x_1) - x_2 + 5)})$, 300 pontos de treino e 20.000 de teste. O gerador é o mesmo, com a mesma semente e as chamadas na mesma ordem.

In [ ]:
def p_verdadeiro(x1, x2):
    return 1.0 / (1.0 + np.exp(-(3.0 * np.sin(x1) - x2 + 5.0)))

rng = np.random.default_rng(7)

def gerar(rng, n):
    x1 = rng.uniform(0, 10, size=n)
    x2 = rng.uniform(0, 10, size=n)
    p = p_verdadeiro(x1, x2)
    y = (rng.uniform(size=n) < p).astype(int)
    return np.column_stack([x1, x2]), y

X_treino, y_treino = gerar(rng, 300)
X_teste, y_teste = gerar(rng, 20_000)

grade_integral = np.linspace(0, 10, 4001)
G1, G2 = np.meshgrid(grade_integral, grade_integral)
Pg = p_verdadeiro(G1, G2)
piso_bayes = float(np.mean(np.minimum(Pg, 1 - Pg)))

X_treino.shape, X_teste.shape, round(piso_bayes * 100, 2)

A taxa de erro de Bayes, a menor taxa de erro de teste esperada que um classificador pode ter, é 13,26%, calculada como na seção 7.7.

A varredura do *k*-NN vai de $k = 1$ a $k = 269$, de quatro em quatro. Ela para em 269 por causa da validação cruzada: com 10 grupos sobre 300 pontos, cada ajuste vê só 270 pontos, e um *k*-NN não tem como consultar mais vizinhos do que o treino tem. Com $k \ge 271$, o `scikit-learn` não levanta exceção; ele avisa, devolve `nan` como pontuação daquele grupo e segue, e um `np.argmin` sobre um array com `nan` devolve a posição do `nan`. O primeiro passo é conferir, só com treino e teste, que os pontos são os mesmos da seção 7.7.

In [ ]:
ks = list(range(1, 270, 4))
knn_treino = []
knn_teste = []
for k in ks:
    modelo = KNeighborsClassifier(n_neighbors=k).fit(X_treino, y_treino)
    knn_treino.append(float(np.mean(modelo.predict(X_treino) != y_treino)))
    knn_teste.append(float(np.mean(modelo.predict(X_teste) != y_teste)))
knn_treino = np.array(knn_treino)
knn_teste = np.array(knn_teste)

k_menor_teste = ks[int(np.argmin(knn_teste))]
len(ks), k_menor_teste, round(float(knn_teste.min()) * 100, 2)

> **🔧 Função**
>
> **`KNeighborsClassifier(n_neighbors)`**: o *k*-NN para resposta qualitativa, que prevê a classe mais comum entre os `n_neighbors` vizinhos mais próximos. Ajusta-se com `.fit(X, y)` e prevê com `.predict(X)`.

São 68 valores de $k$, e o menor erro de teste sai em $k = 9$, com 15,08%: os dois números da seção 7.7. Lá, esse $k = 9$ foi escolhido olhando o próprio conjunto de teste. A pergunta desta seção é qual $k$ a validação cruzada escolhe sem olhar.

### Logística com polinômios

A regressão logística do capítulo 9 separa as classes com uma reta: a fronteira é onde a combinação linear dos preditores zera. A fronteira de Bayes aqui é a curva $x_2 = 3\sin(x_1) + 5$, e uma reta não a acompanha. O remédio é o mesmo da regressão linear: acrescentar potências dos preditores. Com grau 2, o modelo é

$$
\log\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1 X_1 + \beta_2 X_1^2 + \beta_3 X_2 + \beta_4 X_2^2,
$$

e com grau $d$ entram $X_1, \dots, X_1^d$ e $X_2, \dots, X_2^d$. As potências de cada preditor ficam separadas, sem produtos como $X_1 X_2$.

In [ ]:
def logistica_polinomial(grau):
    potencias = ColumnTransformer([
        ("x1", PolynomialFeatures(grau, include_bias=False), [0]),
        ("x2", PolynomialFeatures(grau, include_bias=False), [1]),
    ])
    # C=np.inf: o ajuste sem penalização, como no capítulo 9
    return make_pipeline(
        StandardScaler(), potencias, LogisticRegression(C=np.inf, max_iter=10000)
    )

logistica_polinomial(2).fit(X_treino, y_treino)[:-1].transform(X_treino[:1]).shape

> **🔧 Função**
>
> **`ColumnTransformer([(nome, transformação, colunas), ...])`**: aplica cada transformação só às colunas indicadas e junta os resultados lado a lado. Aqui, a coluna 0 ganha as suas potências e a coluna 1 as dela, e nenhuma potência mistura as duas.
>
> **`LogisticRegression(C, max_iter)`**: a regressão logística.
>
> - `C=np.inf`: sem penalização, o ajuste por máxima verossimilhança do capítulo 9.
> - `max_iter=10000`: o teto de iterações do otimizador, alto para os graus maiores convergirem.
>
> **`pipeline[:-1]`**: o `Pipeline` sem o último passo, só com as transformações.

Com grau 2, o `ColumnTransformer` entrega quatro colunas ao modelo: $X_1$, $X_1^2$, $X_2$ e $X_2^2$, os quatro preditores da fórmula. O `StandardScaler` vem antes, pelo mesmo motivo do capítulo inteiro: potências altas de números entre 0 e 10 ficam em escalas muito diferentes.

In [ ]:
grade_fig = np.linspace(0, 10, 200)
Xg, Yg = np.meshgrid(grade_fig, grade_fig)
pontos_grade = np.column_stack([Xg.ravel(), Yg.ravel()])

graus_fronteira = [1, 2, 3, 4]
modelos_fronteira = [
    logistica_polinomial(grau).fit(X_treino, y_treino) for grau in graus_fronteira
]
erro_teste_fronteira = [
    float(np.mean(m.predict(X_teste) != y_teste)) for m in modelos_fronteira
]
[round(e * 100, 2) for e in erro_teste_fronteira]

In [ ]:
# Figura: Fronteiras de decisão da regressão logística com polinômios de grau 1 a 4 (verde), sobre os 300 pontos de treino coloridos pela classe verdadeira, com a fronteira de Bayes (roxo tracejado). O título de cada painel traz a taxa de erro de teste do ajuste, medida nos 20.000 pontos de teste.
proba_bayes = p_verdadeiro(Xg, Yg)

fig, eixos = plt.subplots(2, 2, figsize=(8, 7.4), sharex=True, sharey=True)
for ax, grau, modelo, erro in zip(
    eixos.ravel(), graus_fronteira, modelos_fronteira, erro_teste_fronteira
):
    proba = modelo.predict_proba(pontos_grade)[:, 1].reshape(Xg.shape)
    ax.scatter(
        X_treino[y_treino == 0, 0], X_treino[y_treino == 0, 1],
        s=12, alpha=0.6, color="C0", label="classe 0",
    )
    ax.scatter(
        X_treino[y_treino == 1, 0], X_treino[y_treino == 1, 1],
        s=12, alpha=0.6, color="C1", label="classe 1",
    )
    ax.contour(Xg, Yg, proba, levels=[0.5], colors="C2", linewidths=2)
    ax.contour(Xg, Yg, proba_bayes, levels=[0.5], colors="C3", linestyles="--", linewidths=1.5)
    ax.set_title(f"grau {grau}: erro de teste {erro * 100:.2f}%".replace(".", ","))
for ax in eixos[1]:
    ax.set_xlabel("x1")
for ax in eixos[:, 0]:
    ax.set_ylabel("x2")
eixos[0, 0].plot([], [], color="C2", linewidth=2, label="fronteira da logística")
eixos[0, 0].plot([], [], color="C3", linestyle="--", linewidth=1.5, label="fronteira de Bayes")
alcas, rotulos = eixos[0, 0].get_legend_handles_labels()
fig.legend(alcas, rotulos, loc="lower center", ncols=2, frameon=False)
plt.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()

> **🔧 Função**
>
> **`modelo.predict_proba(X)`**: a probabilidade estimada de cada classe para cada linha de `X`, uma coluna por classe; `[:, 1]` separa a da classe 1.
>
> **`ax.contour(x, y, z, levels)`**: desenha as curvas onde `z` vale cada número de `levels`. Com `levels=[0.5]`, a curva onde a probabilidade da classe 1 é meio: a fronteira de decisão.
>
> **`fig.legend(alcas, rotulos, loc, ncols)`**: uma legenda só para a figura inteira, fora dos painéis.

A reta do grau 1 erra 21,25% do teste. O grau 2 dobra a fronteira, mas numa parábola que não segue a onda, e o erro cai pouco, para 19,93%. O grau 3 chega a 18,67%, e o grau 4, cuja fronteira já tem as duas curvaturas da de Bayes, a 14,57%. Até que grau vale subir? Com dado real não há 20.000 pontos de teste para responder, e é aí que entra a validação cruzada.

### Treino, teste e validação cruzada

Para cada grau de 1 a 10, três taxas de erro: a de treino, medida nos 300 pontos do ajuste; a de teste, nos 20.000; e a do 10-fold, calculada só com os 300 pontos de treino.

In [ ]:
dez_grupos = StratifiedKFold(10, shuffle=True, random_state=10)

graus = range(1, 11)
log_treino, log_teste, log_cv = [], [], []
for grau in graus:
    modelo = logistica_polinomial(grau).fit(X_treino, y_treino)
    log_treino.append(float(np.mean(modelo.predict(X_treino) != y_treino)))
    log_teste.append(float(np.mean(modelo.predict(X_teste) != y_teste)))
    log_cv.append(
        1 - cross_val_score(logistica_polinomial(grau), X_treino, y_treino, cv=dez_grupos).mean()
    )
log_treino, log_teste, log_cv = np.array(log_treino), np.array(log_teste), np.array(log_cv)

proporcao_classe_1 = float(y_treino.mean())
proporcao_por_grupo = [float(y_treino[validacao].mean()) for _, validacao in dez_grupos.split(X_treino, y_treino)]
round(proporcao_classe_1 * 100, 2), round(min(proporcao_por_grupo) * 100, 2), round(max(proporcao_por_grupo) * 100, 2)

> **🔧 Função**
>
> **`StratifiedKFold(n_splits, shuffle, random_state)`**: o *k*-fold para classificação. Divide as linhas em `n_splits` grupos mantendo, em cada grupo, a proporção de classes do conjunto todo. `shuffle=True` embaralha antes de dividir, e `random_state` é a semente do embaralhamento.
>
> **`esquema.split(X, y)`**: percorre as divisões do esquema; cada uma é um par de arrays de posições, as do treino e as da validação.

Em classificação, `cross_val_score` devolve a **acurácia** de cada grupo, a fração de acertos, e a taxa de erro é `1 -` a média delas.

Dos 300 pontos de treino, 55,67% são da classe 1. Um grupo de 30 pontos não tem como ter exatamente essa proporção, e o `StratifiedKFold` chega o mais perto possível: a proporção da classe 1 em cada grupo fica entre 53,33% e 56,67%, isto é, 16 ou 17 dos 30 pontos. Estratificar evita que um grupo saia, por azar do sorteio, com uma classe sub-representada, o que tornaria o erro daquele grupo pouco parecido com o dos outros. O `cross_val_score` já estratifica sozinho quando recebe um classificador e um número inteiro em `cv`, mas sem embaralhar.

O *k*-NN recebe a mesma validação cruzada, com os mesmos 10 grupos:

In [ ]:
knn_cv = np.array([
    1 - cross_val_score(
        KNeighborsClassifier(n_neighbors=k), X_treino, y_treino, cv=dez_grupos
    ).mean()
    for k in ks
])
bool(np.isnan(knn_cv).any())

Nenhuma pontuação saiu `nan`: com $k$ até 269, todo ajuste tem vizinhos suficientes.

In [ ]:
# Figura: Taxa de erro de treino (azul), de teste (laranja) e do 10-fold (verde) no dado simulado, com a taxa de erro de Bayes (cinza tracejado). Esquerda: regressão logística com polinômios de grau 1 a 10. Direita: k-NN contra 1/k, em escala log, com a vizinhança maior à esquerda. O x marca o mínimo da curva de teste (laranja) e o da curva do 10-fold (verde).
estilos = {
    "treino": {"color": "C0", "label": "erro de treino"},
    "teste": {"color": "C1", "label": "erro de teste"},
    "10-fold": {"color": "C2", "label": "erro do 10-fold"},
}
paineis = [
    (list(graus), {"treino": log_treino, "teste": log_teste, "10-fold": log_cv}),
    (1 / np.array(ks), {"treino": knn_treino, "teste": knn_teste, "10-fold": knn_cv}),
]

fig, eixos = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True)
for ax, (x, curvas) in zip(eixos, paineis):
    for nome, valores in curvas.items():
        ax.plot(x, valores, linewidth=2, **estilos[nome])
        if nome != "treino":
            i = int(np.argmin(valores))
            ax.plot(
                x[i], valores[i], marker="x", markersize=10, markeredgewidth=2.5,
                color=estilos[nome]["color"], linestyle="none", label="_nolegend_",
            )
    ax.axhline(piso_bayes, color="0.45", linestyle="--", linewidth=1.5, label="taxa de erro de Bayes")

eixos[0].set_xticks(list(graus))
eixos[0].set_xlim(0.5, 10.5)
eixos[0].set_xlabel("grau do polinômio")
eixos[0].set_title("regressão logística")
eixos[1].set_xscale("log")
eixos[1].set_xlim(1 / ks[-1] * 0.8, 1.25)
eixos[1].set_xlabel("1/k (escala log; mais flexível →)")
eixos[1].set_title("k-NN")
eixos[0].set_ylim(0, None)
eixos[0].yaxis.set_major_formatter(lambda valor, _: f"{valor:.0%}")
eixos[0].set_ylabel("taxa de erro")
eixos[0].legend()
plt.tight_layout()
plt.show()

Nos dois painéis, as curvas de teste e do 10-fold têm o mesmo desenho: caem enquanto a flexibilidade corrige o viés e param de cair, ou voltam a subir, quando a variância passa a pesar. A curva de treino tem outro desenho, e o fim desta seção volta a ela.

### O que a validação cruzada escolhe

A regra é a de sempre: o grau, ou o $k$, com o menor erro do 10-fold. O chunk abaixo faz a escolha e depois, só para conferir, olha o teste: o erro de teste no escolhido e o menor erro de teste da curva inteira.

In [ ]:
def escolha(opcoes, cv, teste):
    i = int(np.argmin(cv))
    j = int(np.argmin(teste))
    return {
        "escolhido pela CV": opcoes[i],
        "erro do 10-fold nele (%)": round(float(cv[i]) * 100, 2),
        "empates no mínimo": int(np.sum(np.isclose(cv, cv.min()))),
        "erro de teste nele (%)": round(float(teste[i]) * 100, 2),
        "menor erro de teste (%)": round(float(teste[j]) * 100, 2),
        "onde": opcoes[j],
    }

pd.DataFrame.from_dict(
    {
        "logística (grau)": escolha(list(graus), log_cv, log_teste),
        "k-NN (k)": escolha(ks, knn_cv, knn_teste),
    },
    orient="index",
)

A taxa de erro de um grupo de 30 pontos só pode ser um múltiplo de 1/30, e a média dos 10 grupos anda em degraus. Por isso dois graus, ou dois valores de $k$, podem empatar no mínimo; quando empatam, `np.argmin` fica com o primeiro, o menos flexível. Aqui a coluna de empates mostra 1 nos dois métodos: o mínimo é único.

Na logística, o 10-fold escolhe o grau 6, com 17,00% de erro estimado, e o erro de teste do grau 6 é 14,49%, o menor da curva de teste inteira. No *k*-NN, o 10-fold escolhe $k = 13$, também com 17,00%; o erro de teste de $k = 13$ é 15,42%, contra os 15,08% de $k = 9$, o $k$ que a seção 7.7 escolheu olhando o teste. A escolha feita sem tocar no teste custa 15,42 − 15,08 = 0,34 ponto percentual. É o número que faltava à seção 7.7: uma taxa de erro obtida por um procedimento que não usou o conjunto de teste para decidir.

In [ ]:
logistica_escolhida_erra_menos = bool(log_teste[np.argmin(log_cv)] < knn_teste.min())
logistica_escolhida_erra_menos

`logistica_escolhida_erra_menos` confirma que a logística no grau escolhido pelo 10-fold erra menos no teste do que o *k*-NN no melhor $k$ da curva inteira. Compare o lado direito do modelo com polinômios, uma soma de potências de $X_1$ com potências de $X_2$, com o expoente de $\Pr(Y = 1 \mid X = x)$ nesta simulação, $3\sin(x_1) - x_2 + 5$. O que os dois têm em comum, e o que o *k*-NN, que não supõe forma nenhuma, deixa de aproveitar?

### O nível das curvas, e a curva de treino

Na figura, a curva do 10-fold passa por cima da de teste em quase toda a faixa.

In [ ]:
bayes_no_treino = float(np.mean(
    (p_verdadeiro(X_treino[:, 0], X_treino[:, 1]) > 0.5).astype(int) != y_treino
))
(
    int(np.sum(log_cv < log_teste)), len(log_cv),
    int(np.sum(knn_cv < knn_teste)), len(knn_cv),
    round(bayes_no_treino * 100, 2), round(piso_bayes * 100, 2),
)

O 10-fold fica abaixo do teste em nenhum dos 10 graus da logística e em só 4 dos 68 valores de $k$. Nesta amostra, portanto, a validação cruzada **superestima** a taxa de erro de teste. A causa é a que a seção 7.7 encontrou para o erro de treino: a amostra de 300 pontos saiu difícil. Até a regra de Bayes, que conhece $\Pr(Y \mid X)$, erra 16,33% desses 300 pontos, contra 13,26% no quadrado inteiro. Toda taxa medida nesses pontos herda esse azar, e a validação cruzada só tem esses pontos para medir. Com outra amostra de treino, o nível poderia sair abaixo do teste; é a mesma lição dos três cenários da seção 10.3. O nível erra, e o que decide a escolha do modelo é **onde** fica o mínimo.

A curva de treino não serve para isso.

In [ ]:
grau_menor_treino = int(graus[np.argmin(log_treino)])
k_menor_treino = ks[int(np.argmin(knn_treino))]
(
    grau_menor_treino, round(float(log_teste[np.argmin(log_treino)]), 4),
    k_menor_treino, round(float(knn_treino.min()), 4),
    round(float(knn_teste[np.argmin(knn_treino)]), 4),
)

Escolher pelo erro de treino leva ao modelo mais flexível da grade nos dois métodos: o grau 10, com 16,29% de erro de teste, e $k = 1$, que tem 0% de erro de treino (cada ponto é o seu próprio vizinho mais próximo) e 20,92% de erro de teste. O treino mede o quanto o modelo reproduz os pontos que já viu, e quem se dobra mais consegue reproduzi-los melhor, sem que isso diga nada sobre pontos novos.

## Vazamento: o Pré-processamento Dentro da Validação

A validação cruzada estima o erro em observações que o modelo não viu. A estimativa só vale se **nada** do ajuste tiver visto o grupo de validação, e o ajuste é mais que o `fit` do estimador. Tudo o que aprende do dado antes dele também conta: a média e o desvio padrão de uma padronização, a escolha de quais colunas usar. Quando uma dessas etapas é feita no conjunto todo, antes da divisão em grupos, o grupo de validação já influenciou o que o modelo recebe.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import KFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

plt.style.use("estilo-figuras.mplstyle")

auto = pd.read_csv("dados/Auto.csv")
auto["potencia"] = pd.to_numeric(auto["potencia"], errors="coerce")
auto = auto.dropna(subset=["potencia"]).reset_index(drop=True)

> **🔷 Conceito**
>
> **Vazamento** (em inglês, *data leakage*) é a entrada, no ajuste do modelo, de informação das observações que deveriam servir só para medir o erro. Com vazamento, o modelo é julgado em observações que ele, de algum modo, já viu, e a estimativa do erro de teste sai otimista.

### Padronizar antes ou dentro

O *k*-NN mede distância entre observações, e por isso precisa de padronização: sem ela, a coluna de maior escala decide sozinha quem é vizinho de quem, como a seção 7.3 mostrou. Nos dados `Auto`, um *k*-NN com 10 vizinhos prevê `milhas_por_galao` a partir de `potencia`, `peso`, `cilindrada` e `aceleracao`, e o 10-fold mede o erro de três jeitos: sem padronizar, padronizando a tabela toda de uma vez antes da validação cruzada, e com o `StandardScaler` dentro do `Pipeline`.

In [ ]:
X = auto[["potencia", "peso", "cilindrada", "aceleracao"]]
y = auto["milhas_por_galao"]
dez_grupos = KFold(10, shuffle=True, random_state=10)
knn = KNeighborsRegressor(n_neighbors=10)

def mse_cv(modelo, X):
    return float(-cross_val_score(
        modelo, X, y, cv=dez_grupos, scoring="neg_mean_squared_error"
    ).mean())

mse_sem_padronizar = mse_cv(knn, X)

> **🔧 Função**
>
> **`KNeighborsRegressor(n_neighbors)`**: o *k*-NN para resposta numérica, que prevê a média da resposta dos `n_neighbors` vizinhos mais próximos.

O jeito errado padroniza a tabela inteira e só depois entrega o resultado à validação cruzada:

In [ ]:
X_padronizado = StandardScaler().fit_transform(X)
mse_padronizado_antes = mse_cv(knn, X_padronizado)

O jeito certo põe o `StandardScaler` no `Pipeline` e entrega o `Pipeline` inteiro:

In [ ]:
mse_padronizado_dentro = mse_cv(make_pipeline(StandardScaler(), knn), X)

pd.DataFrame(
    {"MSE do 10-fold": [
        round(mse_sem_padronizar, 2),
        round(mse_padronizado_antes, 2),
        round(mse_padronizado_dentro, 2),
    ]},
    index=["sem padronizar", "padronizando antes (errado)", "padronizando dentro (certo)"],
)

A diferença entre os dois códigos é onde o `fit` do `StandardScaler` acontece. No errado, ele acontece uma vez, em todos os carros, e cada grupo de validação é padronizado com uma média e um desvio que ele mesmo ajudou a calcular. No certo, o `cross_val_score` ajusta o `Pipeline` inteiro em cada grupo de treino: o `StandardScaler` aprende a média e o desvio só dos carros de treino, e o grupo de validação é padronizado com esses números, como um carro novo seria.

Padronizar importa: o MSE cai de 17,43, sem padronizar, para 15,06, com o `StandardScaler` dentro, uma diferença de 2,37. **Onde** padronizar importa pouco neste caso: 14,93 no jeito errado contra 15,06 no certo, uma diferença de 0,13. O motivo aparece nos números que o `StandardScaler` aprende.

In [ ]:
treino, validacao = next(dez_grupos.split(X))
print(f"{len(X)} carros: {len(treino)} de treino e {len(validacao)} de validação")
pd.DataFrame({
    f"média, {len(X)} carros": X.mean().round(2),
    f"média, {len(treino)} de treino": X.iloc[treino].mean().round(2),
    f"desvio, {len(X)} carros": X.std().round(2),
    f"desvio, {len(treino)} de treino": X.iloc[treino].std().round(2),
})

`next(dez_grupos.split(X))` pega só a primeira das dez divisões, um par com as posições do treino e as da validação. No primeiro grupo, tirar os 40 carros de validação move a média de `peso` de 2.977,58 para 2.952,62, e a padronização quase não muda. O vazamento existe, e é pequeno porque uma média de centenas de carros mal sente 40 deles. Há etapas em que ele não é pequeno.

### Quando o vazamento decide tudo

Um conjunto de dados fabricado para não ter nada a aprender: 50 observações, 25 de cada classe, e 5.000 colunas de ruído sorteadas de uma normal padrão, sem relação nenhuma com a classe. A classe de uma observação nova não depende de nenhuma das 5.000 colunas, e um classificador qualquer acerta como quem joga uma moeda justa: a taxa de erro de teste é 50%.

In [ ]:
def conjunto_de_ruido(rng):
    X = rng.normal(size=(50, 5000))
    y = rng.permutation(np.repeat([0, 1], 25))
    return X, y

rng = np.random.default_rng(10)
X_ruido, y_ruido = conjunto_de_ruido(rng)
X_ruido.shape, int(y_ruido.sum())

> **🔧 Função**
>
> **`rng.normal(size=(linhas, colunas))`**: uma matriz de sorteios da normal com média 0 e desvio padrão 1, do tamanho pedido.
>
> **`np.repeat([0, 1], 25)`**: 25 zeros seguidos de 25 uns.
>
> **`rng.permutation(arr)`**: uma cópia de `arr` com os elementos numa ordem sorteada. Aqui, embaralha as 50 classes.

Com tantas colunas e tão poucas linhas, um procedimento comum é escolher primeiro as colunas mais promissoras e ajustar o modelo só nelas: as 100 colunas que mais separam as duas classes, e um *k*-NN com um vizinho.

In [ ]:
cinco_grupos = KFold(5, shuffle=True, random_state=10)
vizinho = KNeighborsClassifier(n_neighbors=1)

X_escolhidas = SelectKBest(f_classif, k=100).fit_transform(X_ruido, y_ruido)
erro_errado = 1 - cross_val_score(vizinho, X_escolhidas, y_ruido, cv=cinco_grupos).mean()

> **🔧 Função**
>
> **`SelectKBest(score_func, k)`**: dá a cada coluna uma pontuação calculada por `score_func` e fica com as `k` de pontuação mais alta. Como o `StandardScaler`, aprende com `fit` e aplica com `transform`; `fit_transform` faz os dois de uma vez.
>
> **`f_classif`**: a pontuação usada aqui. Para cada coluna, é maior quanto mais as médias das classes se afastam nela, em relação à dispersão dentro de cada classe.

O jeito certo põe a seleção de colunas dentro do `Pipeline`:

In [ ]:
selecao_e_vizinho = make_pipeline(SelectKBest(f_classif, k=100), vizinho)
erro_certo = 1 - cross_val_score(selecao_e_vizinho, X_ruido, y_ruido, cv=cinco_grupos).mean()

round(float(erro_errado) * 100, 2), round(float(erro_certo) * 100, 2)

Com a seleção feita antes, o 5-fold estima 2% de erro, num problema em que nenhum classificador passa do acaso. Com a seleção dentro do `Pipeline`, estima 48%. As 100 colunas do jeito errado foram escolhidas olhando a classe das 50 observações, inclusive das que depois servem de validação. Entre 5.000 colunas de ruído, algumas separam aquelas 50 observações por puro acaso, e a validação cruzada mede o acerto nesse acaso, que não se repete em observação nova. No jeito certo, cada grupo de treino escolhe as suas próprias 100 colunas sem ver a classe do grupo de validação, e o acaso que ele encontra no treino não vale para a validação.

O desenho deste exemplo vem de Hastie et al. (2009), seção 7.10.2.

### Trinta conjuntos de ruído

Um conjunto só pode ter saído por sorte. A mesma comparação, repetida em 30 conjuntos de ruído, cada um com a sua partição em cinco grupos:

In [ ]:
rng = np.random.default_rng(10)
erros_errados, erros_certos = [], []
for r in range(30):
    X_r, y_r = conjunto_de_ruido(rng)
    grupos = KFold(5, shuffle=True, random_state=r)
    escolhidas = SelectKBest(f_classif, k=100).fit_transform(X_r, y_r)
    erros_errados.append(1 - cross_val_score(vizinho, escolhidas, y_r, cv=grupos).mean())
    erros_certos.append(1 - cross_val_score(selecao_e_vizinho, X_r, y_r, cv=grupos).mean())
erros_errados, erros_certos = np.array(erros_errados), np.array(erros_certos)

todos_errados_abaixo_de_todos_certos = bool(erros_errados.max() < erros_certos.min())
pd.DataFrame(
    {
        "menor (%)": [round(float(e.min()) * 100, 2) for e in (erros_errados, erros_certos)],
        "média (%)": [round(float(e.mean()) * 100, 2) for e in (erros_errados, erros_certos)],
        "maior (%)": [round(float(e.max()) * 100, 2) for e in (erros_errados, erros_certos)],
    },
    index=["seleção antes (errado)", "seleção dentro (certo)"],
), todos_errados_abaixo_de_todos_certos

In [ ]:
# Figura: Taxa de erro do 5-fold em 30 conjuntos de ruído puro (50 observações, 5.000 colunas), com a seleção das 100 colunas feita antes da validação cruzada (laranja) e dentro do Pipeline (azul). Cada ponto é um conjunto; a altura dentro de cada fileira é sorteada só para os pontos não se sobreporem. A linha tracejada marca 50%, a taxa de erro de teste de qualquer classificador nesse dado.
deslocamento = np.random.default_rng(10).uniform(-0.12, 0.12, size=(2, 30))
fileiras = [
    (erros_errados, 1, "C1"),
    (erros_certos, 0, "C0"),
]

fig, ax = plt.subplots(figsize=(8, 3))
for erros, altura, cor in fileiras:
    ax.scatter(erros, altura + deslocamento[altura], s=40, alpha=0.8, color=cor)
ax.axvline(0.5, color="0.45", linestyle="--", linewidth=1.5)
ax.annotate("acaso", xy=(0.5, 1.35), xytext=(6, 0), textcoords="offset points", color="0.45")
ax.set_yticks([0, 1], ["seleção dentro\n(certo)", "seleção antes\n(errado)"])
ax.set_ylim(-0.5, 1.5)
ax.set_xlim(-0.02, 1.0)
ax.xaxis.set_major_formatter(lambda valor, _: f"{valor:.0%}")
ax.set_xlabel("taxa de erro do 5-fold")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`ax.axvline(x, color, linestyle)`**: uma linha vertical na posição `x`, de cima a baixo do gráfico.
>
> **`ax.annotate(texto, xy, xytext, textcoords)`**: escreve `texto` junto ao ponto `xy`; com `textcoords="offset points"`, `xytext` é um deslocamento em pontos tipográficos.
>
> **`ax.set_yticks(posicoes, rotulos)`**: põe marcas no eixo vertical só nas `posicoes` dadas, com os `rotulos` no lugar dos números.

`todos_errados_abaixo_de_todos_certos` confirma que, nos 30 conjuntos, o maior erro estimado com a seleção antes fica abaixo do menor erro estimado com a seleção dentro. O jeito errado vai de 0% a 6%, com média de 1,6%; o certo, de 38% a 72%, com média de 55,47%. O certo não acerta 50% em cada conjunto, e nem deveria: com 50 observações e cinco grupos de 10, a estimativa da validação cruzada oscila muito de uma amostra para outra, como a seção 10.4 mediu. Ela oscila em volta de 50%, onde está o erro verdadeiro. O errado não oscila em volta de nada que exista: ele mede o quanto o acaso das 5.000 colunas consegue separar 50 observações, e esse acaso é grande.

Os dois exemplos cometem o mesmo erro, uma etapa que aprende do dado ajustada fora da validação cruzada. O tamanho do estrago depende do quanto a etapa aprende. A padronização aprende uma média e um desvio por coluna, sem olhar a resposta; a seleção de colunas olha a resposta e escolhe, entre 5.000, as que melhor a acompanham naquelas 50 observações.

> **🔷 Conceito**
>
> Toda etapa que aprende do dado (padronizar, preencher valores faltantes, escolher colunas, reduzir a dimensão) entra no `Pipeline`, e o `Pipeline` inteiro entra no `cross_val_score`. Assim cada etapa é reajustada em cada grupo de treino, e o grupo de validação chega ao modelo como uma observação nova chegaria.

## Leituras adicionais

*A escrever.*

## Referências

- **Hastie; Tibshirani; Friedman**. *The Elements of Statistical Learning*. 2nd ed.. Springer. 2009.
- **James; Witten; Hastie; Tibshirani; Taylor**. *An Introduction to Statistical Learning with Applications in Python*. Springer. 2023.